I am building a custom function that will take the raw sensor data values (unscaled, straight out of the machine), scale them automatically using saved scaler, predict whether a failure is imminent and provide a clear, readable engineering alert

## 🛠️ Operational Inference & Real-Time Custom Prediction Pipeline

To close out the machine learning lifecycle, a modular prediction function `predict_machine_failure()` has been developed. This pipeline acts as a self-contained wrapper that mimics a live production environment.

### ⚙️ Pipeline Flow:
1. **Raw Vector Ingestion:** Accepts real-time, unscaled telemetry streams directly from physical machines.
2. **Deterministic Preprocessing:** Bypasses data leakage hazards by utilizing the serialized `maintenance_scaler.pkl` to scale raw vectors matching the baseline dataset's mean and variance profile.
3. **Probability Threshold Scoring:** Evaluates data via `random_forest_maintenance_model.pkl`, extracting the exact probability margins to gauge confidence scores before emitting structural alert flags.

In [1]:
import pandas as pd
import numpy as np
import joblib

In [10]:
def predict_machine_failure(footfall, AQ, USS, CS, VOC, IP, Temperature):
    # 1. Load the persisted artifacts
    model = joblib.load('random_forest_maintenance_model.pkl')
    scaler = joblib.load('maintenance_scaler.pkl')
    
    # 2. Arrange raw data into a structured DataFrame with explicit feature names
    feature_names = ['footfall', 'AQ', 'USS', 'CS', 'VOC', 'IP', 'Temperature']
    raw_data_df = pd.DataFrame([[footfall, AQ, USS, CS, VOC, IP, Temperature]], columns=feature_names)
    
    # 3. Transform the data point securely without warnings
    scaled_data = scaler.transform(raw_data_df)
    
    # 4. Execute prediction
    prediction = model.predict(scaled_data)[0]
    probabilities = model.predict_proba(scaled_data)[0]
    
    # 5. Compile the diagnostic report
    print("\n================ 🏭 REAL-TIME MACHINE DIAGNOSTIC REPORT ================")
    if prediction == 1:
        print(f"⚠️  STATUS: FAILURE IMMINENT | Risk Level: {probabilities[1]*100:.1f}%")
        print("🛑 ACTION REQUIRED: Flagging dispatch for emergency preventive maintenance.")
    else:
        print(f"✅ STATUS: OPERATING NORMAL  | Health Index: {probabilities[0]*100:.1f}%")
        print("👍 ACTION REQUIRED: Routine monitoring. No immediate adjustments needed.")
    print("========================================================================\n")
    
    return {
        'failed': bool(prediction),
        'failure_probability': float(probabilities[1])
    }

## Testing

In [11]:
print("Testing an operational run:\n")
predict_machine_failure(25,40,500,12.5,0.15,30,45.0)

Testing an operational run:


================ 🏭 REAL-TIME MACHINE DIAGNOSTIC REPORT ================
✅ STATUS: OPERATING NORMAL  | Health Index: 87.0%
👍 ACTION REQUIRED: Routine monitoring. No immediate adjustments needed.



{'failed': False, 'failure_probability': 0.13}

1. Statistical Vetting (Shapiro-Wilk normality testing & Mann-Whitney U group separations)
2. Collinearity Check (Spearman Rank Matrix evaluation)
3. Model Contention & Handling Imbalance (Random Forest vs. Adjusted XGBoost via scale_pos_weight)
4. Local Explainable AI Integration (LIME Black-Box transparency)
5. Model Persistence & Verified Productionization (Joblib compression & a clean custom inference function)